## Build Dataset: Raw Data Acquisition

Pulls each raw series from its source (FRED, Yahoo Finance, BNM, DOSM), saves it to CSV, and collects it into `data_frames` for `02_tranform_dataset.ipynb`.

In [ ]:
import requests
import os
import sys
import time
import pandas as pd
import pickle
import yfinance as yf
from datetime import datetime
from fredapi import Fred
from dotenv import load_dotenv
load_dotenv("../.env")

sys.path.append('..')
from modules.source import Sources, START, END, RAW_DATA_PATH
from modules.helpers import clean_api_response, parse_10y_entry
   
FRED_API_KEY = os.getenv('FRED_API_KEY')
if FRED_API_KEY is None: 
   raise ValueError('FRED_API_KEY not found in .env')

fred = Fred(FRED_API_KEY)
BNM_HEADERS = {'Accept': 'application/vnd.BNM.API.v1+json'}
data_frames: dict[str, pd.DataFrame] = {}

### Palm oil price (FRED)

In [ ]:
series = fred.get_series_first_release(Sources.palm_oil_global.ticker)
df_palm = clean_api_response(series, Sources.palm_oil_global)
df_palm.index = pd.to_datetime(df_palm.index)
df_palm.to_csv(f"{RAW_DATA_PATH}/{Sources.palm_oil_global}.csv")
data_frames[Sources.palm_oil_global] = df_palm

### Fed funds rate (FRED)

In [ ]:
series_upper = fred.get_series('DFEDTARU')
series_lower = fred.get_series('DFEDTARL')

df_ffr = pd.concat([series_upper, series_lower], axis=1, join='inner')
df_ffr.columns = ['Upper', 'Lower']
df_ffr['Midpoint'] = (df_ffr['Upper'] + df_ffr['Lower']) / 2
df_ffr.index.name = 'date'
df_ffr.to_csv(f"{RAW_DATA_PATH}/{Sources.FFR_midpoint}.csv")

data_frames[Sources.FFR_midpoint] = df_ffr[['Midpoint']].rename(columns={'Midpoint': Sources.FFR_midpoint})

### Yahoo Finance series

UST 10Y, VIX, USDMYR, DXY, Brent oil, KLCI.

In [ ]:
yahoo_stocks = [Sources.UST_10Y, Sources.VIX, Sources.USDMYR, Sources.DXY, Sources.brent_oil, Sources.KLCI]

for name in yahoo_stocks:
   df_yf = yf.download(name.ticker, START, END, progress=False)
   try:
      df_yf = clean_api_response(df_yf, name)
   except ValueError as e:
      print(f"  FAILED - {name}: {e}")
      continue

   df_yf.to_csv(f'{RAW_DATA_PATH}/{name}.csv')
   print(f"{name} ({name.ticker}) - saved {len(df_yf)} rows")
   data_frames[name] = df_yf[['Close']].rename(columns={'Close': name})

### CPI inflation, core (DOSM)

In [ ]:
df_cpi_raw = pd.read_parquet('https://storage.dosm.gov.my/cpi/cpi_2d_core_inflation.parquet')
df_cpi_raw.to_csv(f'{RAW_DATA_PATH}/{Sources.cpi_inflation_yoy}.csv')

data_frames[Sources.cpi_inflation_yoy] = (
   df_cpi_raw
   .assign(date=pd.to_datetime(df_cpi_raw['date']))
   .query("division == 'overall'")
   .set_index('date')[['inflation_yoy']]
   .dropna()
   .rename(columns={'inflation_yoy': Sources.cpi_inflation_yoy})
)

### Overnight Policy Rate (BNM)

In [ ]:
records = []
for year in range(START.year, END.year + 1):
   resp = requests.get(f'https://api.bnm.gov.my/public/opr/year/{year}', headers=BNM_HEADERS)
   records.extend(resp.json()['data'])

df_opr = pd.DataFrame(records)
df_opr['date'] = pd.to_datetime(df_opr['date'])
df_opr = df_opr.set_index('date').sort_index()
df_opr = df_opr[~df_opr.index.duplicated(keep='last')]
df_opr.to_csv(f'{RAW_DATA_PATH}/{Sources.OPR}.csv')

data_frames[Sources.OPR] = df_opr.loc[START:END, ['new_opr_level']].rename(columns={'new_opr_level': Sources.OPR})

### MGS 10Y yield (BNM)

BNM only provides daily MGS yield per date request, we will have to make ~4,000 requests. With 0.3 seconds delay per request (to prevent API request overload), it takes roughly 30 minutes to fetch all data. 

In [ ]:
business_days = pd.bdate_range(START, END)  # Mon-Fri only; holidays still queried but expected empty
mgs_records, no_data_days, failed_days = [], [], []

for year, year_group in business_days.groupby(business_days.year):
   year_records = 0
   for d in year_group:
      date_str = d.strftime("%Y-%m-%d")
      resp = requests.get("https://api.bnm.gov.my/public/gov-sec-yield", headers=BNM_HEADERS, params={"date": date_str}, timeout=30)
      if resp.status_code != 200:
         failed_days.append(date_str)
         time.sleep(0.3)
         continue

      entry = parse_10y_entry(resp.json(), date_str)
      if entry is not None:
         mgs_records.append(entry)
         year_records += 1
      else:
         no_data_days.append(date_str)
      time.sleep(0.3)

   print(f"[{year}] {year_records} of {len(year_group)} business days have a 10Y entry")

df_mgs = pd.DataFrame(mgs_records).sort_values("date").reset_index(drop=True)
df_mgs.to_csv(f"{RAW_DATA_PATH}/{Sources.MGS_10Y}.csv", index=False)

df_mgs['date'] = pd.to_datetime(df_mgs['date'])
data_frames[Sources.MGS_10Y] = df_mgs.set_index('date')[['yield_close']].rename(columns={'yield_close': Sources.MGS_10Y})

print(
   f"\nTotal: {len(df_mgs)} daily 10Y observations\n"
   f"Business days with no 10Y entry: {len(no_data_days)}\n"
   f"Failed API calls: {len(failed_days)}\n"
   + (f"\nFailed dates (re-run these individually): {failed_days}" if failed_days else "")
)

### Sector indices (Yahoo Finance)

For each sector, we aggregate 2-3 large, widely-held constituents, and reconstruct its equal-weighted index after saving its raw data to csv. For plantation, we trim data from before 2017-11-30 (SD Gurthie Berhad hasn't listed yet).

In [ ]:
sector_constituents = {
   Sources.financials: ['1155.KL', '1023.KL', '1295.KL'],   # Maybank, CIMB, Public Bank
   Sources.plantation: ['5285.KL', '1961.KL', '2445.KL'],   # Sime Darby Plantation, IOI Corp, KLK
   Sources.REITs:      ['5227.KL', '5176.KL', '5235SS.KL'], # IGB REIT, Sunway REIT, KLCCP Stapled
   Sources.technology: ['0166.KL', '0097.KL', '0128.KL'],   # Inari Amertron, ViTrox, Frontken
   Sources.energy:     ['6033.KL', '5183.KL', '5681.KL'],   # PetGas, PetChem, Petronas Dagangan
   Sources.industrial_products: ['8869.KL', '7113.KL'],     # Press Metal, Top Glove
}

for sector, tickers in sector_constituents.items():
   print(f"\n=== {sector.upper()} ===")
   closing_price_series: dict[pd.Series] = {}
   
   for t in tickers:
      df = yf.download(t, START, END, progress=False)
      try:
         df = clean_api_response(df, t)
      except ValueError as e:
         print(f"  FAILED - {t}: {e}")
         continue

      closing_price_series[t] = df['Close'] # stored as series
      print(f"  OK - {t} ({len(df)} rows)")

   sector_df = pd.DataFrame(closing_price_series)
   sector_df.index.name = 'date'
   sector_df.to_csv(f'{RAW_DATA_PATH}/sectors/{sector}.csv')

   if sector is Sources.plantation:
      sector_df = sector_df.loc[datetime(2017, 11, 30):] # 5285.KL (SD Guthrie Berhad) date of being listed

   sector_returns = sector_df.ffill().pct_change().mean(axis=1)
   sector_index = 100 * (1 + sector_returns.fillna(0)).cumprod()
   data_frames[sector] = pd.DataFrame({sector: sector_index})

Raw collection is done. All data shifting/adjustment, are in `02_tranform_dataset.ipynb`. We just persist `data_frames` for that notebook to pick up.

In [ ]:
with open(f"{RAW_DATA_PATH}/data_frames.pkl", "wb") as f:
   pickle.dump({"data_frames": data_frames}, f)